# DFT XC skeleton 二阶导数分解 (SVWN, LDA)

对标 `06-2-decomp_de_xc_tpss0.ipynb`，但 SVWN 是纯 LDA。LDA 下：

- 只有 RHO 一个分量（无 SIGMA、无 tau）。
- vxc 形状 `[1, ngrids]`，fxc 形状 `[1, 1, ngrids]`。
- ao 二阶导数 (deriv=2) 已足够构造 ipip 部分；diagonal 项也只用到 deriv=2 (XX..ZZ)。

因此 LDA 的 de_xc 实现比 GGA/MGGA 简单许多。


In [1]:
from pyscf import gto, dft, lib
import numpy as np
from functools import partial

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", max_memory=32000).build()

In [3]:
mf = dft.RKS(mol, xc="SVWN").density_fit()
dat0 = np.load("nh3_r_svwn.npz")
mf.mo_coeff = dat0["mo_coeff"]
mf.mo_occ = dat0["mo_occ"]
mf.mo_energy = dat0["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mocc = mo_coeff[:, mo_occ > 0]
dm0 = mocc @ mocc.T * 2
natm = mol.natm
nao = mol.nao
aoslices = mol.aoslice_by_atom()
ni = dft.numint.NumInt()

In [5]:
grids = dft.grid.Grids(mol)
grids.coords = coords = dat0["grid_coords"]
grids.weights = weights = dat0["grid_weights"]
ngrids = len(weights)

In [6]:
# Reference de_vxc from 06-7: this is what we want to reproduce.
de_ks_ref = np.load("nh3_r_svwn_decomp.npz")["de_vxc"]
print("de_vxc_ref shape:", de_ks_ref.shape)
print("de_vxc_ref fp:   ", lib.fp(de_ks_ref))

de_vxc_ref shape: (4, 4, 3, 3)
de_vxc_ref fp:    -1.0550494670106272


In [7]:
# LDA: deriv=2 is enough for diagonal block (uses XX..ZZ)
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho = ni.eval_rho2(mol, ao[0], mo_coeff, mo_occ, xctype="LDA")
print("rho shape:", rho.shape)

rho shape: (43328,)


In [8]:
xc_eff = ni.eval_xc_eff(mf.xc, rho, deriv=2, xctype="LDA")
vxc = xc_eff[1]  # shape [1, ngrid]
fxc = xc_eff[2]  # shape [1, 1, ngrid]
print("vxc shape:", vxc.shape, "fxc shape:", fxc.shape)

vxc shape: (1, 43328) fxc shape: (1, 1, 43328)


In [9]:
TX, TY, TZ = 0, 1, 2
O = 0
X, Y, Z = 1, 2, 3
XX, XY, XZ = 4, 5, 6
YX, YY, YZ = 5, 7, 8
ZX, ZY, ZZ = 6, 8, 9

In [10]:
ao_dm0 = ao @ dm0  # [10, ngrid, nao]
ao_dm0.shape

(10, 43328, 49)

# fxc contribution

In [11]:
# LDA: drho has only 1 component (RHO)
drho = np.zeros((natm, 3, 1, ngrids))
for A in range(natm):
    _, _, p0, p1 = aoslices[A]
    slc = slice(p0, p1)
    DERIV_COMPONENTS = [
        [(TX, 0), (X, O)],
        [(TY, 0), (Y, O)],
        [(TZ, 0), (Z, O)],
    ]
    for ((t, v), (cbra, cket)) in DERIV_COMPONENTS:
        drho[A, t, v] -= np.einsum("gu, gu -> g", ao[cbra, :, slc], ao_dm0[cket, :, slc])
# RHO symmetric coeff: *2
drho *= 2

In [12]:
lib.fp(drho)

np.float64(-39882.21870121027)

In [13]:
de_fxc = np.einsum("g, Atxg, xyg, Bsyg -> ABts", weights, drho, fxc, drho)
print(lib.fp(de_fxc))

-20.132874762943892


In [14]:
# --- dao_vxc_diag (LDA: only XX..ZZ ⊗ rho) --- #
dao_vxc_diag = np.zeros((6, nao))  # 6 denotes xx, xy, xz, yy, yz, zz
wv = weights * vxc  # [1, ngrids]

# Only contribution: ao[i+4]^T @ (wv[0] * ao[0])
aow_diag = np.einsum("gu, g -> gu", ao_dm0[0], wv[0])
for idx, its in enumerate([XX, XY, XZ, YY, YZ, ZZ]):
    dao_vxc_diag[idx] += 2 * np.einsum("gu, gu -> u", ao[its], aow_diag)

de_vxc_diag = np.zeros((natm, natm, 6))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    de_vxc_diag[A, A] += np.einsum("Au -> A", dao_vxc_diag[:, slcA])
de_vxc_diag = de_vxc_diag[:, :, [[0, 1, 2], [1, 3, 4], [2, 4, 5]]]
print("de_vxc_diag fp:", lib.fp(de_vxc_diag))

de_vxc_diag fp: 52.68061529362255


In [15]:
# --- dao_vxc (LDA: only RHO contribution; ipip[t,s] from grad-grad cross) --- #
wv = weights * vxc  # [1, ngrids]
dao_vxc = np.zeros((3, 3, nao, nao))

# LDA: ipip[t,s] += (0.5 * wv[0] * ao[s+1])^T @ ao[t+1]   (×2 from below symmetrization)
aowv = [0.5 * np.einsum("gu, g -> gu", ao[t + 1], wv[0]) for t in range(3)]
for t in range(3):
    for s in range(3):
        dao_vxc[t, s] += 2 * aowv[s].T @ ao[t + 1]

dao_vxc += dao_vxc.transpose(1, 0, 3, 2)  # [s,t] with AO indices transposed

de_vxc_off = np.zeros((natm, natm, 3, 3))
for A in range(natm):
    _, _, p0A, p1A = aoslices[A]
    slcA = slice(p0A, p1A)
    for B in range(A + 1):
        _, _, p0B, p1B = aoslices[B]
        slcB = slice(p0B, p1B)
        de_vxc_off[A, B] += np.einsum("tsuv, uv -> ts", dao_vxc[:, :, slcB, slcA], dm0[slcB, slcA])
        if A != B:
            de_vxc_off[B, A] = de_vxc_off[A, B].T
print("de_vxc fp:", lib.fp(de_vxc_off))

de_vxc fp: -33.60278999768945


In [16]:
de_xc_recap = de_vxc_diag + de_vxc_off + de_fxc
assert np.allclose(de_xc_recap, de_ks_ref)

In [17]:
dat = dict(np.load("nh3_r_svwn_decomp.npz"))
dat.update({
    "de_vxc_diag": de_vxc_diag,
    "de_vxc_off": de_vxc_off,
    "de_fxc": de_fxc,
})
np.savez("nh3_r_svwn_decomp.npz", **dat)